## Read in gTREND nitrogen surplus data at the finest grid

The code below loads in every tif file in the folder Surplus and splits it into two parquet files for storage reasons, one which acts as a lookup (matches pixel id to spatial info) and another that stores the surplus nitrogen data at each pixel. 

We're looking at years 2000-2017. Doing this for all the years created a parquet file that was too large for github, so I split it into 3. Rather than making the code do this, I just manually put the years I wanted for each parquet file in the folder Surplus and then renamed the output file.

In [2]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
import pandas as pd
import numpy as np
from pathlib import Path
from shapely.ops import unary_union
from pyproj import Transformer

# Load all US counties, reproject to match raster
counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip")
counties = counties.to_crs("EPSG:5070")

# Keep only Iowa (STATEFP == "19") and dissolve into one boundary for masking
iowa = counties[counties["STATEFP"] == "19"]
iowa_boundary = [unary_union(iowa.geometry)]

pixel_area_ha = 250 * 250 / 10000  # 6.25 ha

# Transformer from raster CRS (EPSG:5070, meters) to lat/lon (EPSG:4326)
to_latlon = Transformer.from_crs("EPSG:5070", "EPSG:4326", always_xy=True)

# Find all surplus tif files
surplus_dir = Path("../../data/Surplus")
tif_files = sorted(surplus_dir.glob("Surplus_N_*.tif"))
print(f"Found {len(tif_files)} files: {[f.name for f in tif_files]}")

grid_df = None        # static lookup: pixel_id -> x, y, lon, lat (built once)
ref_shape = None      # (height, width) of the cropped Iowa grid, used as a consistency check
ref_transform = None
all_years = []

for tif_path in tif_files:
    year = int(tif_path.stem.split("_")[-1])  # extract year from filename
    print(f"Processing {year}...")

    with rasterio.open(tif_path) as src:
        nodata = src.nodata  # use file's nodata if set, else None
        out_image, out_transform = rio_mask(
            src,
            iowa_boundary,
            crop=True,      # crop to Iowa's bounding box, preserving native 250m grid
            nodata=nodata,  # pixels outside the Iowa polygon get set to nodata
            filled=True
        )

    band = out_image[0]  # single-band raster
    height, width = band.shape

    if ref_shape is None:
        # First file: lock in the reference grid and build the coordinate lookup
        # for EVERY cell in the cropped extent (not just cells valid this year),
        # so pixel_ids line up even if nodata footprints shift across years.
        ref_shape, ref_transform = (height, width), out_transform

        rr, cc = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
        rr, cc = rr.ravel(), cc.ravel()
        full_pixel_id = (rr.astype(np.int64) * width + cc).astype(np.int32)

        xs, ys = rasterio.transform.xy(out_transform, rr, cc)
        xs, ys = np.array(xs), np.array(ys)
        lons, lats = to_latlon.transform(xs, ys)

        grid_df = pd.DataFrame({
            "pixel_id": full_pixel_id,
            "x": xs.astype("float32"),
            "y": ys.astype("float32"),
            "lon": np.asarray(lons, dtype="float32"),
            "lat": np.asarray(lats, dtype="float32"),
        })
    else:
        assert (height, width) == ref_shape, (
            f"{tif_path.name} has a different cropped shape than prior years "
            f"({(height, width)} vs {ref_shape}); pixel_ids would not line up."
        )

    # Identify valid (non-nodata) pixels for this year
    if nodata is not None and not np.isnan(nodata):
        valid_mask = band != nodata
    else:
        valid_mask = ~np.isnan(band)

    rows, cols = np.where(valid_mask)
    values = band[rows, cols].astype("float32")
    pixel_id = (rows.astype(np.int64) * width + cols).astype(np.int32)

    df = pd.DataFrame({
        "pixel_id": pixel_id,
        "year": np.int16(year),
        "surplus_kgha": values,
    })
    df["total_kg_N"] = (df["surplus_kgha"] * pixel_area_ha).astype("float32")
    all_years.append(df)

# Combine into a pixel-level panel (coordinates live separately in grid_df)
panel = pd.concat(all_years, ignore_index=True)

grid_df.to_parquet("iowa_grid_lookup.parquet", index=False)
panel.to_parquet("nitrogen_surplus_iowa_grid_panel.parquet", index=False, compression="snappy")

print(f"Grid cells (lookup table): {len(grid_df):,}")
print(f"Panel rows (pixel x year): {len(panel):,}")
print(panel.head())

Found 0 files: []


ValueError: No objects to concatenate

## Data Exploration

Check that the data looks as expected

In [3]:
import pandas as pd

grid = pd.read_parquet("iowa_grid_lookup.parquet")
panel = pd.read_parquet("nitrogen_surplus_iowa_grid_panel.parquet")

print("grid rows:", len(grid), "| pixel_id dtype:", grid["pixel_id"].dtype)
print("panel rows:", len(panel), "| pixel_id dtype:", panel["pixel_id"].dtype)
print("years available:", sorted(panel["year"].unique()))
print("lat range:", grid["lat"].min(), "to", grid["lat"].max())
print("lon range:", grid["lon"].min(), "to", grid["lon"].max())

year = 2005  # use one of the years printed above
sub = panel[panel["year"] == year]
print("rows for that year:", len(sub))


merged = sub.merge(grid, on="pixel_id", how="inner")
print("rows after merge:", len(merged))
print(merged.sample(10))

FileNotFoundError: [Errno 2] No such file or directory: 'iowa_grid_lookup.parquet'


Test what the data looks like for a single year by creating a static render. Creating a widget was too much for the junyper notebook to handle with all of the points.

In [17]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # forces a purely static renderer -- no zoom/pan, no widget
import matplotlib.pyplot as plt

GRID_PATH = "iowa_grid_lookup.parquet"
PANEL_PATH = "nitrogen_surplus_iowa_grid_panel.parquet"


def plot_iowa_year_static(
    year,
    value_col="surplus_kgha",
    grid_path=GRID_PATH,
    panel_path=PANEL_PATH,
    cmap="YlOrRd",
    clip_quantiles=(0.01, 0.99),
    use_latlon=True,
    out_path=None,
    dpi=150,
):
    """
    Render the full-resolution Iowa grid for one year as a single static image.

    year:            the year to plot
    value_col:       column to color by, e.g. "surplus_kgha" or "total_kg_N"
    clip_quantiles:  clip the color scale to these percentiles
    use_latlon:      label/extent the plot in lon/lat (default) vs the native
                      projected x/y (meters, EPSG:5070). Shape detection always
                      uses the native x/y grid since that's the one guaranteed
                      to be perfectly rectangular; lon/lat is used only for the
                      displayed extent, which is a very close approximation for
                      an area the size of Iowa.
    out_path:        if given, saves the figure to this path (e.g. "iowa_2015.png")
    """
    grid = pd.read_parquet(grid_path)
    panel = pd.read_parquet(panel_path, filters=[("year", "==", year)])
    if panel.empty:
        raise ValueError(f"No rows found for year={year} in {panel_path}")

    # Recover the grid's (height, width). pixel_id was built as row*width + col,
    # and every column shares one native x value on a north-up, unrotated raster,
    # so the count of distinct x values is exactly the width.
    width = grid["x"].nunique()
    height = len(grid) // width
    if height * width != len(grid):
        raise ValueError(
            "Grid lookup table doesn't reshape into a clean rectangle -- "
            "double check iowa_grid_lookup.parquet wasn't modified unexpectedly."
        )

    # Rebuild the 2D array via flat indexing: pixel_id IS the row-major flat
    # index into a (height, width) array, so no python loop is needed at all.
    img = np.full(height * width, np.nan, dtype="float32")
    img[panel["pixel_id"].to_numpy()] = panel[value_col].to_numpy()
    img = img.reshape(height, width)

    if use_latlon:
        x_min, x_max = grid["lon"].min(), grid["lon"].max()
        y_min, y_max = grid["lat"].min(), grid["lat"].max()
        xlabel, ylabel = "Longitude", "Latitude"
    else:
        x_min, x_max = grid["x"].min(), grid["x"].max()
        y_min, y_max = grid["y"].min(), grid["y"].max()
        xlabel, ylabel = "X (m, EPSG:5070)", "Y (m, EPSG:5070)"

    vmin, vmax = np.nanquantile(img, clip_quantiles)

    fig, ax = plt.subplots(figsize=(8, 10), dpi=dpi)
    im = ax.imshow(
        img,
        extent=[x_min, x_max, y_min, y_max],
        origin="upper",  # row 0 of the array is the top (max lat / max y)
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        aspect="equal",
    )
    ax.set_title(f"Nitrogen surplus — Iowa, {year}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Surplus (kg N/ha)" if value_col == "surplus_kgha" else value_col)
    fig.tight_layout()

    if out_path:
        fig.savefig(out_path, dpi=dpi)
        print(f"Saved to {out_path}")

    return fig


if __name__ == "__main__":
    plot_iowa_year_static(2005, out_path="iowa_surplus_2005_static.png")

Saved to iowa_surplus_2005_static.png
